# Kaggle T4×2 Qualification — Higgs TTS 3 4B Production Showcase
## Reviewer-facing bilingual acceptance notebook / Notebook nghiệm thu song ngữ cho reviewer

**Release under test:** annotated Git tag `v1.0.0`  
**Hardware:** Kaggle NVIDIA Tesla T4 ×2  
**Model input:** `dangkhoa2016/bosonai-higgs-tts-3-4b/transformers/default/1`

### English
This notebook is the public, reviewer-facing entry point for the qualified Higgs TTS T4×2 deployment. It favors observable evidence over hidden automation: source provenance, hardware/model preflight, runtime versions, service readiness, four synthetic reference voice previews plus eight longer, individually explained listening demonstrations, the official warm benchmark rerun, and a final machine-readable acceptance record.

### Tiếng Việt
Notebook này là entry point công khai dành cho reviewer của cấu hình Higgs TTS đã qualify trên T4×2. Notebook ưu tiên evidence quan sát được thay vì automation ẩn: provenance của source, preflight phần cứng/model, phiên bản runtime, readiness của service, bốn synthetic reference voice preview cùng tám demo nghe thử dài hơn, được giải thích riêng, rerun benchmark warm chính thức và bản nghiệm thu machine-readable cuối cùng.


## Before you run / Trước khi chạy

### English
In Kaggle Notebook settings:
1. Import this notebook from `dangkhoa2016/Kaggle-T4x2-Qualification-for-Higgs-TTS-3-4B`.
2. Select **GPU T4 x2**.
3. Turn **Internet ON** so the notebook can clone the immutable public release tag.
4. Use **Add Input → Models** and attach `dangkhoa2016/bosonai-higgs-tts-3-4b`, variation `transformers/default`, Version `1`.
5. Start from a fresh session and use **Run All**.

The notebook uses only the attached Kaggle model for weights. It does not download model weights from Hugging Face, does not use GGUF, and does not perform real-person voice cloning.

### Tiếng Việt
Trong phần Settings của Kaggle Notebook:
1. Import notebook này từ `dangkhoa2016/Kaggle-T4x2-Qualification-for-Higgs-TTS-3-4B`.
2. Chọn **GPU T4 x2**.
3. Bật **Internet ON** để notebook clone immutable public release tag.
4. Dùng **Add Input → Models** và attach `dangkhoa2016/bosonai-higgs-tts-3-4b`, variation `transformers/default`, Version `1`.
5. Bắt đầu từ fresh session và chọn **Run All**.

Notebook chỉ dùng Kaggle Model đã attach cho model weights. Notebook không tải model weights từ Hugging Face, không dùng GGUF và không thực hiện clone giọng người thật.


## Architecture / Kiến trúc

```text
CPU   -> preprocessing
GPU0  -> tts_engine
GPU1  -> audio_encoder + vocoder
TP    -> disabled
```

### English
The accepted runtime contract uses original weights in FP16, CUDA graphs disabled, `max_running_requests=1`, `mem_fraction_static=0.72`, PyTorch sampling, and Triton attention. The service is intentionally validated as a local production-style HTTP boundary before benchmark reproduction.

### Tiếng Việt
Runtime contract đã được chấp nhận dùng original weights ở FP16, tắt CUDA graphs, `max_running_requests=1`, `mem_fraction_static=0.72`, sampling bằng PyTorch và attention bằng Triton. Service được kiểm tra như một HTTP boundary kiểu production trên localhost trước khi rerun benchmark.


## Step 1 — Immutable release bootstrap / Bootstrap release bất biến

### English
Kaggle imports only the notebook file, not the full repository. This step clones annotated tag `v1.0.0`, proves the tag object is annotated, and proves that the peeled tag target equals the checked-out commit. The notebook deliberately prints the resolved release commit instead of embedding a self-referential expected SHA.

### Tiếng Việt
Kaggle chỉ import file notebook chứ không import toàn bộ repository. Bước này clone annotated tag `v1.0.0`, chứng minh tag object là annotated tag và chứng minh peeled tag target đúng bằng commit đang checkout. Notebook cố ý in resolved release commit thay vì hard-code một expected SHA tự tham chiếu.


In [ ]:
from pathlib import Path
import hashlib, json, os, platform, shutil, subprocess, sys, time

# Kaggle GPU images may expose the NVIDIA driver libraries under /usr/local/nvidia/lib64
# without registering that directory in the dynamic loader path for notebook subprocesses.
# Establish both runtime and linker search paths before the first nvidia-smi/CUDA probe.
NVIDIA_DRIVER_LIB = "/usr/local/nvidia/lib64"
for _var in ("LD_LIBRARY_PATH", "LIBRARY_PATH"):
    _old = os.environ.get(_var, "")
    _parts = [x for x in _old.split(":") if x]
    if NVIDIA_DRIVER_LIB not in _parts:
        os.environ[_var] = NVIDIA_DRIVER_LIB + ((":" + _old) if _old else "")
print("NVIDIA_DRIVER_LIB_PATH=PASS", NVIDIA_DRIVER_LIB)

RELEASE_TAG = "v1.0.0"
REPO_URL = "https://github.com/dangkhoa2016/Kaggle-T4x2-Qualification-for-Higgs-TTS-3-4B.git"
ROOT = Path("/kaggle/working/Kaggle-T4x2-Qualification-for-Higgs-TTS-3-4B")
MODEL = Path("/kaggle/input/models/dangkhoa2016/bosonai-higgs-tts-3-4b/transformers/default/1")
VENV = Path("/tmp/higgs-tts-v1.0.0-venv312")
ACCEPT = Path("/kaggle/working/higgs-tts-v1.0.0-acceptance")
ACCEPT.mkdir(parents=True, exist_ok=True)

def run(cmd, **kwargs):
    print("$", cmd if isinstance(cmd, str) else " ".join(map(str, cmd)), flush=True)
    return subprocess.run(cmd, check=True, text=True, **kwargs)

if ROOT.exists():
    shutil.rmtree(ROOT)
run(["git", "clone", "--branch", RELEASE_TAG, "--depth", "1", REPO_URL, str(ROOT)])
actual = subprocess.check_output(["git", "-C", str(ROOT), "rev-parse", "HEAD"], text=True).strip()
tag = subprocess.check_output(["git", "-C", str(ROOT), "describe", "--tags", "--exact-match", "HEAD"], text=True).strip()
tag_ref = f"refs/tags/{RELEASE_TAG}"
tag_type = subprocess.check_output(["git", "-C", str(ROOT), "cat-file", "-t", tag_ref], text=True).strip()
tag_target = subprocess.check_output(["git", "-C", str(ROOT), "rev-parse", tag_ref + "^{}"], text=True).strip()
print("RELEASE_TAG=", tag)
print("RELEASE_TAG_OBJECT_TYPE=", tag_type)
print("RELEASE_COMMIT=", actual)
assert tag == RELEASE_TAG
assert tag_type == "tag", f"Expected annotated tag, got {tag_type!r}"
assert tag_target == actual, (tag_target, actual)
print("RELEASE_PROVENANCE=PASS")


## Step 2 — Kaggle T4×2 and attached-model preflight / Preflight T4×2 và model đã attach

### English
Before installing or loading the TTS runtime, fail closed unless the session exposes at least two Tesla T4 GPUs, `uv`, Python 3.12, and the exact Kaggle Model Version 1 mount. This keeps configuration mistakes separate from model/runtime failures.

### Tiếng Việt
Trước khi cài hoặc load TTS runtime, notebook fail-closed nếu session không có ít nhất hai Tesla T4, `uv`, Python 3.12 và đúng mount của Kaggle Model Version 1. Cách này tách lỗi cấu hình khỏi lỗi model/runtime.


In [ ]:
print("=== KAGGLE HARDWARE / MODEL PREFLIGHT ===")
print("notebook_python:", sys.version.replace("\n", " "))
print("platform:", platform.platform())
assert Path("/kaggle/working").is_dir(), "This notebook is intended for Kaggle"
assert shutil.which("uv"), "Kaggle image must provide uv"
assert Path("/usr/bin/python3.12").is_file(), "Qualified runtime requires /usr/bin/python3.12"
run(["nvidia-smi", "--query-gpu=index,name,memory.total,driver_version", "--format=csv,noheader"])
gpu_names = subprocess.check_output(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], text=True).strip().splitlines()
assert len(gpu_names) >= 2, f"Need Kaggle T4 x2; found {len(gpu_names)} GPU(s): {gpu_names}"
assert all("T4" in name for name in gpu_names[:2]), f"Expected first two GPUs to be T4: {gpu_names}"
assert MODEL.is_dir(), "Attach Kaggle model Version 1 first: " + str(MODEL)
weight_files = sorted(MODEL.glob("*.safetensors"))
assert weight_files, f"No safetensors weights found under {MODEL}"
print("GPU_COUNT=", len(gpu_names))
print("GPU_NAMES=", gpu_names)
print("MODEL_ATTACHMENT=PASS", MODEL)
print("SAFETENSORS_FILES=", len(weight_files))


## Step 3 — Qualified Python 3.12 runtime / Runtime Python 3.12 đã qualify

### English
Create a clean virtual environment and install the repository's frozen runtime lock with `uv`. This is intentionally separate from Kaggle's notebook Python so the service runs on the qualified Python 3.12 dependency set.

### Tiếng Việt
Tạo virtual environment sạch và cài frozen runtime lock của repository bằng `uv`. Runtime này được tách khỏi Python của notebook Kaggle để service chạy đúng dependency set Python 3.12 đã qualify.


In [ ]:
print("=== CREATE QUALIFIED PYTHON 3.12 RUNTIME ===")
if VENV.exists():
    shutil.rmtree(VENV)
run(["uv", "venv", "--python", "/usr/bin/python3.12", "--seed", str(VENV)])
py = VENV / "bin/python"
run([str(py), "--version"])
run(["uv", "pip", "install", "--python", str(py), "-r", str(ROOT / "production/requirements-runtime-lock.txt")])
print("RUNTIME_INSTALL=PASS")


## Step 4 — Runtime provenance / Provenance runtime

### English
Print the versions that materially define the qualified stack and capture a pre-load GPU snapshot. Reviewers can compare these values with the repository runtime documentation before any inference occurs.

### Tiếng Việt
In các phiên bản định nghĩa trực tiếp qualified stack và chụp GPU snapshot trước khi load model. Reviewer có thể đối chiếu các giá trị này với tài liệu runtime của repository trước khi inference bắt đầu.


In [ ]:
print("=== RUNTIME VERSION PROVENANCE ===")
probe = """
import json, sys, torch
import sglang, sglang_omni, flashinfer
print(json.dumps({
  'python': sys.version.split()[0],
  'torch': torch.__version__,
  'torch_cuda': torch.version.cuda,
  'sglang': getattr(sglang, '__version__', 'unknown'),
  'sglang_omni': getattr(sglang_omni, '__version__', 'unknown'),
  'flashinfer': getattr(flashinfer, '__version__', 'unknown'),
}, indent=2))
"""
run([str(VENV / "bin/python"), "-c", probe])
run(["nvidia-smi", "--query-gpu=index,name,memory.used,memory.free,utilization.gpu", "--format=csv,noheader,nounits"])
print("RUNTIME_PROVENANCE=PASS")

print("=== PREWARM SGLANG FLASHINFER SAMPLING JIT ===")
jit_env = os.environ.copy()
jit_env["FLASHINFER_WORKSPACE_BASE"] = str(Path.home() / ".cache/sglang")
run([
    str(VENV / "bin/python"),
    "-c",
    "from flashinfer.sampling import get_sampling_module; "
    "print('FLASHINFER_SAMPLING_PREWARM_START', flush=True); "
    "get_sampling_module(); "
    "print('FLASHINFER_SAMPLING_PREWARM=PASS', flush=True)",
], env=jit_env)


## Step 5 — Start the real T4×2 service / Khởi động T4×2 service thật

### English
Launch the same repository production helper used by the qualified path. FlashInfer cold-JIT requires the observed CUDA linker path, so `LIBRARY_PATH` is set explicitly. Server log and PID evidence are redirected into the acceptance directory.

### Tiếng Việt
Khởi động đúng production helper của repository đã dùng trong qualified path. FlashInfer cold-JIT cần CUDA linker path đã quan sát được, vì vậy `LIBRARY_PATH` được set tường minh. Server log và PID evidence được ghi vào thư mục acceptance.


In [ ]:
print("=== START REAL T4x2 SERVICE ===")
env = os.environ.copy()
env.update({
    "ROOT": str(ROOT),
    "HIGGS_VENV": str(VENV),
    "HIGGS_MODEL_PATH": str(MODEL),
    "HIGGS_LOG_FILE": str(ACCEPT / "server.log"),
    "HIGGS_PID_FILE": str(ACCEPT / "server.pid"),
    "LIBRARY_PATH": "/usr/local/nvidia/lib64:" + env.get("LIBRARY_PATH", ""),
})
run(["bash", str(ROOT / "production/start-background.sh")], env=env)
print("SERVER_START_REQUESTED=PASS")


## Step 6 — Wait for real readiness / Chờ readiness thật

### English
Poll `/health` until the service reports a healthy running state. Cold model load/JIT can take time; the cell prints the final health payload and, on failure, exposes the server-log tail instead of hiding the root cause.

### Tiếng Việt
Poll `/health` cho tới khi service báo trạng thái healthy và running. Cold model load/JIT có thể mất thời gian; cell in final health payload và nếu fail sẽ hiện tail của server log thay vì che nguyên nhân.


In [ ]:
print("=== WAIT FOR /health ===")
import urllib.request
health_url = "http://127.0.0.1:8000/health"
deadline = time.time() + 20 * 60
last_error = None
while time.time() < deadline:
    try:
        with urllib.request.urlopen(health_url, timeout=5) as r:
            health = json.loads(r.read().decode())
            if r.status == 200 and health.get("status") == "healthy" and health.get("running") is True:
                print(json.dumps(health, indent=2))
                print("SERVER_READY=PASS")
                break
    except Exception as exc:
        last_error = repr(exc)
    time.sleep(10)
else:
    log = ACCEPT / "server.log"
    if log.exists():
        print("--- server.log tail ---")
        print("\n".join(log.read_text(errors="replace").splitlines()[-120:]))
    raise RuntimeError(f"Server did not become healthy within 20 minutes. last_error={last_error}")


## Step 7 — Readiness contract and smoke request / Contract readiness và smoke request

### English
Run the public readiness helper and one real production smoke request. A passing health endpoint alone is not enough; the smoke request must return a valid WAV through the actual HTTP synthesis path.

### Tiếng Việt
Chạy public readiness helper và một production smoke request thật. Chỉ health endpoint PASS là chưa đủ; smoke request phải trả về WAV hợp lệ qua HTTP synthesis path thật.


In [ ]:
print("=== RELEASE READINESS + SMOKE ===")
run(["bash", str(ROOT / "production/readiness.sh")], env=env)
smoke_env = env.copy()
smoke_env["HIGGS_SMOKE_OUTPUT"] = str(ACCEPT / "production-smoke.wav")
run([str(VENV / "bin/python"), str(ROOT / "production/smoke-request.py")], env=smoke_env)
print("SMOKE_TEST=PASS")


## Reviewer-facing voice showcase / Showcase giọng nói cho reviewer

### English
This expanded showcase separates **zero-shot baselines** from **synthetic-reference conditioned voices**. The notebook first creates four synthetic reference voice candidates (A–D) inside this same Kaggle session using only the attached model; no real-person recordings are used. The reference clips are playable so reviewers can verify that the candidate voices are audibly different before judging the longer demonstrations.

Eight main demonstrations follow. They are intentionally longer than the smoke test and cover English, Vietnamese, code-switching, onset behavior, and long-form narration. Each main case validates HTTP 200, mono 24 kHz WAV output, minimum listening duration, latency/RTF, SHA-256, conditioning mode, and voice profile. Automatic transport checks do **not** replace human listening review.

### Tiếng Việt
Showcase mở rộng này tách **zero-shot baseline** khỏi các giọng được condition bằng **synthetic reference**. Notebook trước tiên tự tạo bốn synthetic reference voice candidate A–D ngay trong Kaggle session bằng chính model đã attach; không dùng recording của người thật. Bốn reference clip có audio player để reviewer xác nhận chúng thực sự nghe khác nhau trước khi đánh giá các demo dài.

Sau đó là tám demo chính dài hơn smoke test, gồm English, Vietnamese, code-switching, onset behavior và long-form narration. Mỗi case xác minh HTTP 200, WAV mono 24 kHz, minimum listening duration, latency/RTF, SHA-256, conditioning mode và voice profile. Automatic transport checks **không** thay thế human listening review.


In [ ]:
from IPython.display import Audio, display
from time import perf_counter
import wave

SHOWCASE_EXPECTED_CASES = 8
SHOWCASE_MAX_NEW_TOKENS = 1024
AUDIO_ACTIVITY_RMS_THRESHOLD = 0.005
AUDIO_ACTIVITY_MIN_RATIO = 0.55
AUDIO_ACTIVITY_MAX_SILENCE_S = 3.0
AUDIO_ACTIVITY_WINDOW_S = 0.25
SYNTHETIC_VOICE_PROFILES = {
    "VOICE_A": {
        "seed": 21001,
        "text": "This synthetic reference voice is created only for reproducible model evaluation on Kaggle. It does not imitate or identify any real person.",
    },
    "VOICE_B": {
        "seed": 32002,
        "text": "A second synthetic test voice is generated from the model itself, giving reviewers another reference timbre for controlled listening comparisons.",
    },
    "VOICE_C": {
        "seed": 43003,
        "text": "Đây là giọng tham chiếu tổng hợp thứ ba, được tạo hoàn toàn bởi mô hình để phục vụ kiểm thử có thể tái lập và không sao chép giọng người thật.",
    },
    "VOICE_D": {
        "seed": 54004,
        "text": "Giọng tổng hợp thứ tư dùng làm tham chiếu cho phần kể chuyện dài, giúp reviewer so sánh một cấu hình giọng khác trong cùng phiên chạy.",
    },
}
synthetic_voice_results = {}
showcase_results = {}

def _write_and_validate_wav(body, out):
    out.write_bytes(body)
    with wave.open(str(out), "rb") as w:
        duration = w.getnframes() / w.getframerate()
        sr = w.getframerate()
        ch = w.getnchannels()
    return duration, sr, ch

def _audio_activity_metrics(out):
    with wave.open(str(out), "rb") as w:
        assert w.getsampwidth() == 2 and w.getnchannels() == 1, {"sample_width": w.getsampwidth(), "channels": w.getnchannels()}
        sr = w.getframerate()
        raw = w.readframes(w.getnframes())
    from array import array
    import math
    samples = array("h")
    samples.frombytes(raw)
    if sys.byteorder != "little":
        samples.byteswap()
    win = max(1, int(sr * AUDIO_ACTIVITY_WINDOW_S))
    rms = []
    for start in range(0, len(samples), win):
        block = samples[start:start + win]
        if not block:
            continue
        rms.append(math.sqrt(sum(v * v for v in block) / len(block)) / 32768.0)
    silent = [value < AUDIO_ACTIVITY_RMS_THRESHOLD for value in rms]
    longest = current = 0
    for flag in silent:
        current = current + 1 if flag else 0
        longest = max(longest, current)
    trailing = 0
    for flag in reversed(silent):
        if not flag:
            break
        trailing += 1
    active_ratio = 1.0 - (sum(silent) / len(silent)) if silent else 0.0
    return {
        "active_audio_ratio": active_ratio,
        "longest_near_silence_s": longest * AUDIO_ACTIVITY_WINDOW_S,
        "trailing_near_silence_s": trailing * AUDIO_ACTIVITY_WINDOW_S,
        "activity_rms_threshold": AUDIO_ACTIVITY_RMS_THRESHOLD,
        "activity_window_s": AUDIO_ACTIVITY_WINDOW_S,
    }

def _speech_request(payload, out, timeout=240):
    req = urllib.request.Request(
        "http://127.0.0.1:8000/v1/audio/speech",
        data=json.dumps(payload, ensure_ascii=False).encode(),
        headers={"Content-Type": "application/json"}, method="POST")
    t0 = perf_counter()
    with urllib.request.urlopen(req, timeout=timeout) as resp:
        body = resp.read(); status = resp.status
    latency = perf_counter() - t0
    duration, sr, ch = _write_and_validate_wav(body, out)
    return body, status, latency, duration, sr, ch

def create_synthetic_reference_voice(profile_id):
    profile = SYNTHETIC_VOICE_PROFILES[profile_id]
    out = ACCEPT / f"synthetic-{profile_id}.wav"
    payload = {
        "input": profile["text"], "seed": profile["seed"],
        "temperature": 0.8, "top_k": 50, "max_new_tokens": 512,
    }
    body, status, latency, duration, sr, ch = _speech_request(payload, out)
    row = {
        "voice_profile": profile_id, "reference_text": profile["text"], "reference_seed": profile["seed"],
        "http_status": status, "latency_s": latency, "duration_s": duration,
        "sample_rate": sr, "channels": ch, "sha256": hashlib.sha256(body).hexdigest(), "path": str(out),
        **_audio_activity_metrics(out),
    }
    assert status == 200 and sr == 24000 and ch == 1 and duration > 2.0, row
    synthetic_voice_results[profile_id] = row
    print(json.dumps(row, ensure_ascii=False, indent=2))
    display(Audio(filename=str(out)))
    return row

for _profile_id in SYNTHETIC_VOICE_PROFILES:
    create_synthetic_reference_voice(_profile_id)
assert len(synthetic_voice_results) == 4
print("SYNTHETIC_REFERENCE_VOICES=4/4")

SHOWCASE_CASES = {
    "EN01": {
        "language": "English", "conditioning_mode": "zero_shot", "voice_profile": "ZERO_SHOT",
        "seed": 12345, "minimum_duration_s": 10.0,
        "text": "Welcome to this independent Kaggle acceptance run for Higgs TTS. We are testing the public release on two NVIDIA Tesla T4 GPUs, using the original model weights and the documented production layout. This longer English sample gives reviewers enough time to judge clarity, pacing, pronunciation, sentence transitions, and whether the voice remains stable across a complete paragraph.",
    },
    "VI01": {
        "language": "Vietnamese", "conditioning_mode": "zero_shot", "voice_profile": "ZERO_SHOT",
        "seed": 12345, "minimum_duration_s": 10.0,
        "text": "Xin chào, đây là phần kiểm tra tiếng Việt zero-shot của Higgs TTS trên hai GPU T4 của Kaggle. Đoạn này kiểm tra từ đầu câu, thanh điệu, nhịp đọc và sự ổn định của giọng trong một mẫu có độ dài vừa phải.",
    },
    "MIX01": {
        "language": "Vietnamese → English", "conditioning_mode": "synthetic_reference", "voice_profile": "VOICE_B",
        "seed": 12345, "minimum_duration_s": 10.0,
        "text": "Hôm nay chúng ta đang chạy một bài acceptance test có đầy đủ release provenance trên Kaggle với hai GPU T4. The important question is whether the same synthetic reference voice can remain coherent when the sentence moves naturally from Vietnamese into English, then returns to tiếng Việt mà không làm mất từ đầu câu hoặc thay đổi giọng một cách đột ngột.",
    },
    "ONSET01": {
        "language": "Vietnamese onset focus", "conditioning_mode": "synthetic_reference", "voice_profile": "VOICE_A",
        "seed": 12346, "minimum_duration_s": 9.0,
        "text": "Việc kiểm tra phần mở đầu câu rất quan trọng vì các thử nghiệm trước từng phát hiện hiện tượng mất từ đầu tiên. Vì vậy đoạn này cố ý bắt đầu rõ ràng, tiếp tục qua nhiều mệnh đề và kết thúc bằng một câu đầy đủ để reviewer có thể nghe xem onset, khoảng nghỉ và phần âm thanh cuối có được giữ nguyên hay không.",
    },
    "EN02": {
        "language": "English", "conditioning_mode": "synthetic_reference", "voice_profile": "VOICE_B",
        "seed": 12347, "minimum_duration_s": 12.0,
        "text": "This second English demonstration uses a synthetic reference voice produced earlier in the same notebook. A useful production showcase should make it possible to compare more than one speaking identity, so this paragraph is deliberately long enough to expose changes in timbre, rhythm, articulation, and breath-like pauses. Reviewers should compare it with the zero-shot English baseline rather than relying on a single short sentence.",
    },
    "VI02": {
        "language": "Vietnamese", "conditioning_mode": "synthetic_reference", "voice_profile": "VOICE_C",
        "seed": 12348, "minimum_duration_s": 12.0,
        "text": "Demo tiếng Việt thứ hai sử dụng một giọng tham chiếu tổng hợp khác, được tạo ngay trong notebook này. Đoạn văn dài hơn giúp người nghe so sánh màu giọng, cách phát âm dấu, tốc độ hội thoại và độ ổn định khi câu có nhiều cụm từ liên tiếp. Nếu giọng bị thay đổi giữa chừng, mất âm tiết hoặc tạo khoảng lặng bất thường thì reviewer sẽ dễ nhận ra hơn so với một sample chỉ kéo dài vài giây.",
    },
    "MIX02": {
        "language": "English → Vietnamese", "conditioning_mode": "synthetic_reference", "voice_profile": "VOICE_C",
        "seed": 12349, "minimum_duration_s": 12.0,
        "text": "This demonstration begins in English so we can evaluate a clean onset with the conditioned synthetic voice. Sau đó nội dung chuyển sang tiếng Việt để kiểm tra khả năng code-switching trong cùng một lượt sinh âm thanh. The final sentence returns to English and gives the reviewer enough continuous audio to judge whether speaker identity and pacing remain consistent across both languages.",
    },
    "NARR01": {
        "language": "Long-form bilingual narration", "conditioning_mode": "synthetic_reference", "voice_profile": "VOICE_D",
        "seed": 12350, "minimum_duration_s": 20.0,
        "text": "This story begins at the end of a long engineering session, when a small team reviewed the evidence one more time. They checked the release tag, the model source, the GPU layout, and every benchmark record before calling the run complete. Ngoài cửa sổ, thành phố đã lên đèn, nhưng trong notebook vẫn còn một việc quan trọng: nghe lại toàn bộ đoạn âm thanh đủ dài để phát hiện những lỗi mà một sample ngắn có thể che giấu. The goal of this narration is therefore simple: preserve a stable synthetic voice, natural pacing, clear transitions, and intelligible speech from the first word to the final sentence.",
    },
}
assert len(SHOWCASE_CASES) == SHOWCASE_EXPECTED_CASES

def display_showcase_case(case_id):
    case = SHOWCASE_CASES[case_id]
    payload = {
        "input": case["text"], "seed": case["seed"], "temperature": 0.8,
        "top_k": 50, "max_new_tokens": 1024,
    }
    if case["conditioning_mode"] == "synthetic_reference":
        voice = synthetic_voice_results[case["voice_profile"]]
        payload["references"] = [{
            "audio_path": voice["path"],
            "text": voice["reference_text"],
        }]
    out = ACCEPT / f"{case_id}.wav"
    body, status, latency, duration, sr, ch = _speech_request(payload, out)
    row = {
        "id": case_id, "language": case["language"], "text": case["text"], "seed": case["seed"],
        "conditioning_mode": case["conditioning_mode"], "voice_profile": case["voice_profile"],
        "http_status": status, "latency_s": latency, "duration_s": duration, "rtf": latency / duration,
        "sample_rate": sr, "channels": ch, "sha256": hashlib.sha256(body).hexdigest(), "path": str(out),
        **_audio_activity_metrics(out),
    }
    assert status == 200 and sr == 24000 and ch == 1 and duration >= case["minimum_duration_s"], row
    showcase_results[case_id] = row
    print(json.dumps(row, ensure_ascii=False, indent=2))
    display(Audio(filename=str(out)))
    return row

print("SHOWCASE_HELPERS=READY")


### Sample 1 — English natural acceptance / Sample tiếng Anh

Zero-shot English baseline. This is intentionally longer than the smoke request and uses no reference voice.


In [ ]:
display_showcase_case("EN01")


### Sample 2 — Vietnamese zero-shot / Sample tiếng Việt zero-shot

Zero-shot Vietnamese baseline with a deliberately moderate length. A previous long-form run produced speech for only the first sentence followed by a long silent tail, so this rerun records waveform activity metrics in addition to transport and duration.


In [ ]:
display_showcase_case("VI01")


### Sample 3 — Vietnamese → English code-switch / Code-switch Việt → Anh

Conditioned with **Synthetic Voice B**. Spoken technical text is normalized to “hai GPU T4” instead of the literal token `T4 x2`, because the previous run skipped verbalizing `x2`. The technical metadata elsewhere in the notebook still uses T4 x2.


In [ ]:
display_showcase_case("MIX01")


### Sample 4 — Onset-focused Vietnamese / Tiếng Việt tập trung onset

Conditioned with **Synthetic Voice A** and designed specifically to expose missing-initial-word behavior.


In [ ]:
display_showcase_case("ONSET01")


### Sample 5 — English with Synthetic Voice B / Tiếng Anh với Synthetic Voice B

A second long English reading, now reference-conditioned, for direct comparison with the EN01 zero-shot baseline.


In [ ]:
display_showcase_case("EN02")


### Sample 6 — Vietnamese with Synthetic Voice C / Tiếng Việt với Synthetic Voice C

A different synthetic reference profile and a longer Vietnamese paragraph for timbre and tone review.


In [ ]:
display_showcase_case("VI02")


### Sample 7 — English → Vietnamese code-switch with Synthetic Voice C

The same conditioned profile crosses English → Vietnamese → English to expose identity or pacing drift.


In [ ]:
display_showcase_case("MIX02")


### Sample 8 — Long-form narration with Synthetic Voice D / Kể chuyện dài với Synthetic Voice D

The longest reviewer sample. Its opening was changed from “At the end...” to “This story begins...” after a human review found that the previous generation omitted the initial word “At”. The sample remains intentionally long to reveal drift, truncation, unstable pacing, or voice changes.


In [ ]:
display_showcase_case("NARR01")


## Step 8 — Showcase evidence table / Bảng evidence của showcase

### English
Consolidate all eight executed main cases and the four generated synthetic reference voices into machine-readable JSON. In addition to HTTP/audio-format checks, every main WAV is analyzed in 250 ms windows with a normalized RMS threshold of `0.005`. The automatic activity gate requires at least 55% active windows and rejects any near-silence run longer than 3 seconds. This catches silent-tail failures but does **not** prove text coverage; first-word omissions and skipped tokens still require human listening.

### Tiếng Việt
Gộp cả tám main case và bốn synthetic reference voice vào JSON machine-readable. Ngoài HTTP/audio-format gate, mỗi WAV chính được phân tích theo cửa sổ 250 ms với normalized RMS threshold `0.005`. Activity gate tự động yêu cầu ít nhất 55% cửa sổ có hoạt động âm thanh và reject đoạn near-silence dài hơn 3 giây. Gate này bắt được lỗi silent-tail nhưng **không** chứng minh text coverage; mất từ đầu câu hoặc bỏ token vẫn cần human listening.


In [ ]:
import pandas as pd
expected_ids = ["EN01", "VI01", "MIX01", "ONSET01", "EN02", "VI02", "MIX02", "NARR01"]
assert len(expected_ids) == SHOWCASE_EXPECTED_CASES
assert list(showcase_results) == expected_ids, (list(showcase_results), expected_ids)
results = [showcase_results[k] for k in expected_ids]
assert len(results) == SHOWCASE_EXPECTED_CASES
assert all(r["http_status"] == 200 and r["sample_rate"] == 24000 and r["channels"] == 1 for r in results)
(ACCEPT / "synthetic-reference-voices.json").write_text(json.dumps(synthetic_voice_results, ensure_ascii=False, indent=2))
(ACCEPT / "showcase-suite.json").write_text(json.dumps(results, ensure_ascii=False, indent=2))
for r in results:
    r["audio_activity_pass"] = (
        r["active_audio_ratio"] >= AUDIO_ACTIVITY_MIN_RATIO
        and r["longest_near_silence_s"] <= AUDIO_ACTIVITY_MAX_SILENCE_S
    )
summary_rows = [{k:r[k] for k in ("id","language","conditioning_mode","voice_profile","seed","http_status","latency_s","duration_s","rtf","sample_rate","channels","active_audio_ratio","longest_near_silence_s","trailing_near_silence_s","audio_activity_pass","sha256")} for r in results]
display(pd.DataFrame(summary_rows))
print(f"SHOWCASE_HTTP_SUCCESS={sum(r['http_status'] == 200 for r in results)}/{len(results)}")
print(f"SMALL_ACCEPTANCE_HTTP_SUCCESS={sum(r['http_status'] == 200 for r in results)}/{len(results)}")  # legacy reviewer marker
print(f"AUDIO_ACTIVITY_PASS={sum(r['audio_activity_pass'] for r in results)}/{len(results)}")
print("AUDIO_PREVIEW_READY=PASS")


## Step 9 — Official warm benchmark reproduction / Tái lập benchmark warm chính thức

### English
Run the repository's official sequential warm benchmark entrypoint in a fresh evidence directory. The frozen benchmark protocol remains **11 cases × 3 repetitions = 33 requests** and is deliberately separate from the eight reviewer demos above. A fresh rerun is comparison evidence and never silently replaces the retained published benchmark authority.

### Tiếng Việt
Chạy official sequential warm benchmark entrypoint của repository vào evidence directory mới. Frozen benchmark protocol vẫn giữ nguyên **11 case × 3 repetition = 33 request** và tách biệt với tám reviewer demo phía trên. Rerun mới chỉ là evidence để đối chiếu và không âm thầm thay thế retained published benchmark authority.


In [ ]:
print("=== OFFICIAL BENCHMARK REPRODUCTION ===")
bench_dir = ACCEPT / "official-benchmark-rerun"
if bench_dir.exists():
    shutil.rmtree(bench_dir)
run(["bash", str(ROOT / "production/run-official-benchmark.sh"), str(bench_dir)], env=env)
benchmark_summary = json.loads((bench_dir / "summary.json").read_text())
print(json.dumps(benchmark_summary, indent=2))
assert benchmark_summary["status"] == "PASS", benchmark_summary
assert benchmark_summary["requests_total"] == 33, benchmark_summary
assert benchmark_summary["requests_http200"] == 33, benchmark_summary
assert benchmark_summary["cases"] == 11, benchmark_summary
assert benchmark_summary["repetitions"] == 3, benchmark_summary
print("OFFICIAL_BENCHMARK_RERUN=PASS")


## Step 10 — Final machine-readable acceptance / Nghiệm thu machine-readable cuối

### English
The final record separates transport, waveform activity, and human perceptual review. `machine_transport_acceptance` covers the 8/8 showcase requests, 4/4 synthetic references, valid WAV format, and 33/33 benchmark rerun. `audio_activity_acceptance` fails on long silent tails. `perceptual_acceptance` intentionally remains `PENDING_HUMAN_REVIEW` because RMS cannot detect a missing first word or a skipped alphanumeric token. The legacy top-level `acceptance` is PASS only when transport **and** audio-activity gates pass.

### Tiếng Việt
Final record tách transport, waveform activity và human perceptual review. `machine_transport_acceptance` bao phủ 8/8 showcase request, 4/4 synthetic reference, WAV hợp lệ và benchmark rerun 33/33. `audio_activity_acceptance` sẽ FAIL khi có silent-tail dài. `perceptual_acceptance` luôn giữ `PENDING_HUMAN_REVIEW` vì RMS không thể phát hiện mất từ đầu tiên hoặc bỏ alphanumeric token. Top-level `acceptance` cũ chỉ PASS khi cả transport và audio-activity gate cùng PASS.


In [ ]:
showcase_http_ok = len(results) == SHOWCASE_EXPECTED_CASES and all(r["http_status"] == 200 for r in results)
showcase_audio_ok = all(r["sample_rate"] == 24000 and r["channels"] == 1 and r["duration_s"] > 0 for r in results)
audio_activity_ok = all(
    r["active_audio_ratio"] >= AUDIO_ACTIVITY_MIN_RATIO
    and r["longest_near_silence_s"] <= AUDIO_ACTIVITY_MAX_SILENCE_S
    for r in results
)
reference_ok = len(synthetic_voice_results) == 4 and all(r["http_status"] == 200 for r in synthetic_voice_results.values())
benchmark_ok = (
    benchmark_summary["status"] == "PASS"
    and benchmark_summary["requests_total"] == 33
    and benchmark_summary["requests_http200"] == 33
    and benchmark_summary["cases"] == 11
    and benchmark_summary["repetitions"] == 3
)
machine_transport_ok = showcase_http_ok and showcase_audio_ok and reference_ok and benchmark_ok
machine_pass = machine_transport_ok and audio_activity_ok
assert machine_pass, {
    "showcase_http_ok": showcase_http_ok, "showcase_audio_ok": showcase_audio_ok,
    "audio_activity_ok": audio_activity_ok, "reference_ok": reference_ok, "benchmark_ok": benchmark_ok,
}
final = {
    "release_tag": RELEASE_TAG,
    "release_commit": actual,
    "release_tag_object_type": tag_type,
    "model_version": "dangkhoa2016/bosonai-higgs-tts-3-4b/transformers/default/1",
    "gpu_layout": "CPU preprocessing; GPU0 tts_engine; GPU1 audio_encoder+vocoder; TP disabled",
    "server_ready": True, "smoke_test": True,
    "showcase_http_success": f"{sum(r['http_status'] == 200 for r in results)}/{len(results)}",
    "synthetic_reference_voice_success": f"{sum(r['http_status'] == 200 for r in synthetic_voice_results.values())}/{len(synthetic_voice_results)}",
    "synthetic_reference_voices": synthetic_voice_results,
    "showcase_cases": results,
    "benchmark_summary": benchmark_summary,
    "machine_transport_acceptance": "PASS" if machine_transport_ok else "FAIL",
    "audio_activity_acceptance": "PASS" if audio_activity_ok else "FAIL",
    "perceptual_acceptance": "PENDING_HUMAN_REVIEW",
    "voice_clone_production_claim": "PENDING_HUMAN_ACCEPTANCE",
    "acceptance": "PASS" if machine_pass else "FAIL",
}
(ACCEPT / "FINAL-ACCEPTANCE.json").write_text(json.dumps(final, ensure_ascii=False, indent=2))
print(json.dumps(final, ensure_ascii=False, indent=2))
print("ACCEPTANCE=PASS" if machine_pass else "ACCEPTANCE=FAIL")
print("Evidence directory:", ACCEPT)


## Step 11 — Optional cleanup / Cleanup tùy chọn

### English
The default reviewer workflow leaves the service alive after acceptance so logs and outputs remain inspectable. Set `RUN_CLEANUP=True` only after review if you want this notebook to stop the owned background service.

### Tiếng Việt
Mặc định notebook giữ service chạy sau acceptance để reviewer còn xem log và output. Chỉ set `RUN_CLEANUP=True` sau khi review nếu muốn notebook dừng background service thuộc phiên chạy này.


In [ ]:
RUN_CLEANUP = False
if RUN_CLEANUP:
    stop_script = ROOT / "production/stop-background.sh"
    if stop_script.is_file():
        run(["bash", str(stop_script)], env=env)
        print("CLEANUP=PASS")
    else:
        print("CLEANUP=SKIPPED stop-background.sh not present")
else:
    print("CLEANUP=SKIPPED (reviewer default)")


## Reproducibility and limitations / Tái lập và giới hạn

### English
- The notebook proves one qualified Kaggle Tesla T4×2 configuration, not universal T4 compatibility.
- Single-T4 inference was functionally possible but failed the retained production-headroom qualification after warm/reference use.
- The official benchmark authority remains the frozen repository evidence record; this notebook's 33/33 rerun is comparison evidence, not an automatic replacement.
- Request-level universal bitwise determinism is not claimed. Cross-run waveform hashes may differ even when each run is internally stable.
- Four **synthetic reference voice candidates** are generated by the model itself with different fixed seeds. They are labeled A–D until a human reviewer confirms that their perceptual timbres are sufficiently distinct; the notebook does not fabricate gender or speaker-identity labels.
- Waveform activity checks detect long silent tails, but they do not prove that every input token was spoken. Human listening remains required for onset omissions, skipped technical tokens, pronunciation, and voice quality.
- Voice-clone runtime transport with synthetic references is exercised, but the production claim remains `PENDING_HUMAN_ACCEPTANCE`. No real-person voice cloning is used.
- The retained onset classification remains `zero-shot AR acoustic-generation/bootstrap instability before codec decode`; the exact learned-model mechanism remains unproven.

### Tiếng Việt
- Notebook chứng minh một cấu hình Kaggle Tesla T4×2 đã qualify, không tuyên bố mọi T4 đều tương thích.
- Single-T4 inference chạy được về chức năng nhưng không đạt retained production-headroom qualification sau warm/reference use.
- Official benchmark authority vẫn là frozen evidence record trong repository; rerun 33/33 của notebook chỉ là comparison evidence, không tự động thay thế authority.
- Không tuyên bố universal request-level bitwise determinism. Waveform hash giữa các run có thể khác ngay cả khi từng run tự nó ổn định.
- Bốn **synthetic reference voice candidate** được chính model tạo bằng các fixed seed khác nhau. Chúng chỉ được gọi A–D cho tới khi human reviewer xác nhận perceptual timbre thực sự đủ khác; notebook không tự gắn nhãn giới tính hoặc speaker identity.
- Waveform activity check phát hiện silent-tail dài nhưng không chứng minh mọi input token đã được đọc. Human listening vẫn cần cho onset omission, technical token bị bỏ qua, phát âm và chất lượng giọng.
- Runtime transport cho voice clone bằng synthetic reference được exercise, nhưng production claim vẫn là `PENDING_HUMAN_ACCEPTANCE`. Không dùng clone giọng người thật.
- Retained onset classification vẫn là `zero-shot AR acoustic-generation/bootstrap instability before codec decode`; exact learned-model mechanism vẫn chưa được chứng minh.
